# Sandbox for testing new updated preprocessing pipeline
Just using this as a space to try out different aspects of a new preprocessing pipeline using updated spikeinterface and kilsosort4, plus testing out quad base probes  
Use the updated_si_preprocess as your kernel

In [1]:
# import everything we're currently using in our preprocessing pipeline
import os
import sys
import json
import logging
import numpy as np
import scipy.io as scio
import argparse
import shutil
from datetime import datetime
import spikeinterface.full as si
import time
import hdf5storage
import gc

In [2]:
print(si.get_global_job_kwargs())  # get the global job kwargs for parallel processing

{'pool_engine': 'process', 'n_jobs': 1, 'chunk_duration': '1s', 'progress_bar': True, 'mp_context': None, 'max_threads_per_worker': 1}


In [3]:
CATGT_OUTPUT_DIR = os.path.abspath('D:/quadBaseTest/CatGT_New/catgt_fullTest_QuadBase_g0')
STREAM_NAME = f"imec0.ap"
output_folder = os.path.abspath("D:/preprocessing_sandbox")
temp_folder = os.path.join(output_folder, "temp")

In [4]:
# load in the raw recording from the CatGT output directory. Got rid of load_sync_channels - legacy
raw_rec = si.read_spikeglx(CATGT_OUTPUT_DIR, stream_id=STREAM_NAME)
raw_rec

SpikeGLXRecordingExtractor: 1536 channels - 30.0kHz - 1 segments - 162,112,202 samples 
                            5,403.74s (1.50 hours) - int16 dtype - 463.81 GiB

In [5]:
# confirm that probe information is present
raw_rec.get_probe().to_dataframe()

,x,y,contact_shapes,width,shank_ids,contact_ids
0,0.0,0.0,square,12.0,0,s0e0
1,32.0,0.0,square,12.0,0,s0e1
2,0.0,15.0,square,12.0,0,s0e2
3,32.0,15.0,square,12.0,0,s0e3
4,0.0,30.0,square,12.0,0,s0e4
...,...,...,...,...,...,...
1531,782.0,2835.0,square,12.0,3,s3e379
1532,750.0,2850.0,square,12.0,3,s3e380
1533,782.0,2850.0,square,12.0,3,s3e381
1534,750.0,2865.0,square,12.0,3,s3e382


# Pre-process and extract the LFP data 

## Filter LFP band first
Usually we detect bad channels then we filter. But, if your recording isn't filtered, the bad channel detection will high-pass filter the recording for you. For LFP data, we probably want ot filter at the lower bandpass frequency, so we'll filter first. That being said, the new version of spikeinterface literally won't let you filter with a highpass too low because the data is processed in chunks and that introduces some artifacts. Here's some important information   

### Chunk
"A “chunk” is a piece of recording that gets processed in parallel by SpikeInterface. The default chunk duration for most operations is 1 second, but we’ll see how this is not adequate for LFP processing."  

### Margin 
"When we apply a filter on chunked data, we extract additional "margins" of traces at the chunk borders. This is done to reduce border artifacts"

[https://spikeinterface.readthedocs.io/en/stable/forhowto/plot_extract_lfps.html#](https://spikeinterface.readthedocs.io/en/stable/forhowto/plot_extract_lfps.html#)

In [ ]:
# Change the chunk_duration to 30s before filtering, otherwise I get a warning about the margin being too big 
# relative to the chunk size. It's a fake warning because filtering doesn't actually happen until I save it
# but still, I don't like the aesthetics of the warning.
si.set_global_job_kwargs(chunk_duration='30s', n_jobs=1, progress_bar=True)  # set the global job kwargs for parallel processing
print(si.get_global_job_kwargs())  # confirm the change to the global job kwargs for parallel processing

{'pool_engine': 'process', 'n_jobs': 1, 'chunk_duration': '30s', 'progress_bar': True, 'mp_context': None, 'max_threads_per_worker': 1}


In [ ]:
# Filter the LPF recording from 1-400 Hz with the recommended large chunks, set above (30-60s), and 
# large margins (5 seconds). Theoretically the margin_ms should be automatically calculated as 5 seconds, 
# but this just hard codes it.
margin_ms = 5000 # 5 seconds
chunk_duration = "30s" # 30 seconds

t_start = time.perf_counter()
lfpFilteredRec = si.bandpass_filter(
    raw_rec, 
    freq_min=1.0, 
    freq_max=400.0, 
    margin_ms=margin_ms,
    ignore_low_freq_error=True
)

t_end = time.perf_counter()
print(f"LFP filtering time: {t_end - t_start:.2f} seconds")

LFP filtering time: 0.32 seconds


In [8]:
# Should first identify how many groups of channels are present in the recording.
# If multiple groups (multiple shanks), then we should split the recoring and preprocess by group
split_rec = lfpFilteredRec.split_by("group")


In [9]:
# detect bad channels shank-wise. The documentation says that the neighborhood_r2 method is 
# the most robust for bad channel detection for lfp, so we will use that.
bad_ids = []
for shank, sub_rec in split_rec.items():
    print(f"Detecting bad channels for shank {shank}...")
    ids, labels = si.detect_bad_channels(sub_rec, method="neighborhood_r2")
    print(f"Bad channels for shank {shank}: {ids}")
    bad_ids.extend(ids)

Detecting bad channels for shank 0...
Bad channels for shank 0: ['imec0.ap#AP0' 'imec0.ap#AP1' 'imec0.ap#AP2' 'imec0.ap#AP3'
 'imec0.ap#AP4' 'imec0.ap#AP5' 'imec0.ap#AP6' 'imec0.ap#AP7'
 'imec0.ap#AP8' 'imec0.ap#AP9' 'imec0.ap#AP10' 'imec0.ap#AP11'
 'imec0.ap#AP12' 'imec0.ap#AP13' 'imec0.ap#AP14' 'imec0.ap#AP15'
 'imec0.ap#AP16' 'imec0.ap#AP17' 'imec0.ap#AP18' 'imec0.ap#AP19'
 'imec0.ap#AP20' 'imec0.ap#AP21' 'imec0.ap#AP22' 'imec0.ap#AP23'
 'imec0.ap#AP24' 'imec0.ap#AP25' 'imec0.ap#AP26' 'imec0.ap#AP27'
 'imec0.ap#AP28' 'imec0.ap#AP29' 'imec0.ap#AP30' 'imec0.ap#AP31'
 'imec0.ap#AP32' 'imec0.ap#AP33' 'imec0.ap#AP34' 'imec0.ap#AP35'
 'imec0.ap#AP36' 'imec0.ap#AP37' 'imec0.ap#AP38' 'imec0.ap#AP39'
 'imec0.ap#AP40' 'imec0.ap#AP41' 'imec0.ap#AP42' 'imec0.ap#AP43'
 'imec0.ap#AP44' 'imec0.ap#AP45' 'imec0.ap#AP46' 'imec0.ap#AP47'
 'imec0.ap#AP48' 'imec0.ap#AP49' 'imec0.ap#AP50' 'imec0.ap#AP51'
 'imec0.ap#AP52' 'imec0.ap#AP53' 'imec0.ap#AP54' 'imec0.ap#AP55'
 'imec0.ap#AP56' 'imec0.ap#AP57' 'im

In [9]:
# All channels are bad in this example recording because the probe wasn't in the brain, 
# so create a new list of bad channels to check bad channel rejection
bad_ids = ['imec0.ap#AP0', 'imec0.ap#AP1','imec0.ap#AP412', 'imec0.ap#AP680', 'imec0.ap#AP998','imec0.ap#AP999']
if isinstance(bad_ids, np.ndarray):
    bad_ids = bad_ids.tolist()

In [10]:
# reject the bad channels. remove_channels() fails for the split 
# recording dictionary, but .detect_and_remove_bad_channels is built 
# for split recordings and won't redo detection if you give it bad 
# channels, it will only remove them
split_rec_clean = si.detect_and_remove_bad_channels(split_rec, bad_channel_ids=bad_ids)

In [11]:
# downsample the recording to 2500 Hz for LFP analysis. This is done after filtering to avoid aliasing.
lfp_downsampled = si.resample(split_rec_clean, resample_rate=2500, margin_ms=margin_ms)

In [12]:
# re-aggregate the recording so that the existing pipeline's code works without modification
lfp_rec = si.aggregate_channels(lfp_downsampled)

In [13]:
# working on figuring out if all of the split recording segments are properly filtered
# spikeinterface doesn't normally properly propogate the is_filtered annotation when aggregating split recordings
filtered_flags = [sub_rec.get_annotation("is_filtered") for sub_rec in lfp_downsampled.values()]

if all(filtered_flags):
    print("All split recordings are filtered.")
    lfp_rec.annotate(is_filtered=True)
    print("Changed the annotation for the lfp_recording to is_filtered=True.")
else:
    print("Some split recordings are not filtered. Please check the annotations.")


All split recordings are filtered.
Changed the annotation for the lfp_recording to is_filtered=True.


In [14]:
params = {}
params['INTER_SHANK_SPACE'] = 250  # in microns - distance between shanks for Neuropixels 2.0
params['LFP_SPACING'] = 100  # in microns - distance between channels to be saved for LFP analysis

# select channels for saving
keep_idx = []
for iShank in range(4):  # 4 shanks for Neuropixels 2.0
    idx_depth = [(idx, x[1]) for idx, x in enumerate(lfp_rec.get_channel_locations()) 
                    if int(x[0] // params['INTER_SHANK_SPACE']) == iShank]
    
    # If no channels for this shank, skip
    if len(idx_depth) == 0:
        continue
        
    depths = [x[1] for x in idx_depth]
    idx = [x[0] for x in idx_depth]
    
    # EFBG - new method of getting queried_depths, accounting for recordings where multiple different portions of a single shank are recorded (e.g. dorsal and ventral hc)
    depths_array = np.array(depths)
    sorted_depths = np.sort(depths_array)
    gap_threshold = 100; # in microns - Threshold to identify gaps in the depth distribution (in microns) - theoretically channels should only be 15 microns apart, but made larger in case of bad channels
    depth_diffs = np.diff(sorted_depths)
    gap_indices = np.where(np.abs(depth_diffs) > gap_threshold)[0]
    # If there are gaps, split the depths into segments
    if len(gap_indices) > 0:
        queried_depths = []
        start_idx = 0
        for gap_idx in gap_indices:
            segment_depths = sorted_depths[start_idx:gap_idx+1]
            segment_queried = np.arange(min(segment_depths), max(segment_depths), params['LFP_SPACING'])
            queried_depths.extend(segment_queried)
            start_idx = gap_idx + 1
        # Add the last segment
        if start_idx < len(sorted_depths):
            segment_depths = sorted_depths[start_idx:]
            segment_queried = np.arange(min(segment_depths), max(segment_depths), params['LFP_SPACING'])
            queried_depths.extend(segment_queried)
    else:
        # No gaps, use all depths
        queried_depths = np.arange(min(depths), max(depths), params['LFP_SPACING']).tolist()

    
    # Add the max depth if not already there
    if queried_depths[-1] < max(depths) - 50:
        queried_depths.append(max(depths))
        
    # Find the closest channels to the requested depths
    for b in queried_depths:
        min_diff = float('inf')
        closest_index = None
        for i, a in enumerate(depths):
            diff = abs(a - b)
            if diff < min_diff:
                min_diff = diff
                closest_index = i
            elif diff == min_diff:
                closest_index = min(closest_index, i)
        # Only add a channel if not already present - EFBG added
        if idx[closest_index] not in keep_idx:
            keep_idx.append(idx[closest_index])
        
# Get the selected channels
final_channels = lfp_rec.channel_ids[np.asarray(keep_idx)]

# Get channel locations and calculate shank IDs
all_locs = lfp_rec.get_channel_locations()
final_depths = [(all_locs[idx][1] + 175) for idx in keep_idx] # EFBG - adding 175 microns such that the tip of the probe is at 0 microns depth (for NPX2)
shank_ids = [int(all_locs[idx][0] // params['INTER_SHANK_SPACE']) for idx in keep_idx]

In [16]:
print(lfp_rec.save.__doc__)


        Save a SpikeInterface object.

        Parameters
        ----------
        kwargs: Keyword arguments for saving.
            * format: "memory", "zarr", or "binary" (for recording) / "memory" or "numpy_folder" or "npz_folder" for sorting.
                In case format is not memory, the recording is saved to a folder. See format specific functions for
                more info (`save_to_memory()`, `save_to_folder()`, `save_to_zarr()`)
            * folder: if provided, the folder path where the object is saved
            * name: if provided and folder is not given, the name of the folder in the global temporary
                    folder (use set_global_tmp_folder() to change this folder) where the object is saved.
              If folder and name are not given, the object is saved in the global temporary folder with
              a random string
            * dump_ext: "json" or "pkl", default "json" (if format is "folder")
            * verbose: if True output is verbose

In [15]:
# save the recording - part 1 (saving the selected channels to a new recording)
# Create a temporary folder for faster extraction
temp_lfp_folder = temp_folder
os.makedirs(temp_lfp_folder, exist_ok=True)

# Extract LFP data using a temporary binary file for speed
job_kwargs = dict(n_jobs=6, chunk_duration=chunk_duration, progress_bar=True)

temp_lfp = lfp_rec.select_channels(channel_ids=final_channels)


In [16]:
# save the recording continued. should now actually only save the selected channels, and not all channels.

temp_lfp_save = temp_lfp.save(
            folder=temp_lfp_folder,
            format="binary", 
            return_scaled=True, 
            cast_unsigned=True,
            overwrite=True,
            **job_kwargs
        )


write_binary_recording 
engine=process - n_jobs=6 - samples_per_chunk=75,000 - chunk_memory=17.02 MiB - total_memory=102.14 MiB - chunk_duration=30.00s


write_binary_recording (workers: 6 processes spawn):   0%|          | 0/181 [00:00<?, ?it/s]

In [17]:
# Get traces from the temporary recording
final_lfp = temp_lfp_save.get_traces(channel_ids=final_channels, return_in_uV=True) # got rid of cast_unsigned=True, #return_scaled is also deprecated, use retun_in_uV instead

# Get other metadata
final_tvec = lfp_rec.get_times()
final_tvec = final_tvec - final_tvec[0] # because spikeinterface adds the "First Sample's time"
final_fs = lfp_rec.get_sampling_frequency()

In [18]:
# Save LFP times to the CatGT output folder in both formats
lfp_times_npy_path = os.path.join(CATGT_OUTPUT_DIR, f"imec0_lfp_times.npy")
np.save(lfp_times_npy_path, final_tvec)

This next code block always fails in both things I attempt to do. When I try to clean up the temporary folder I always get the exception and I'm told that it can't remove it. Then, when I try to save LFP data to a MATLAB folder, that fails and gives me an OverflowError, related to scio.savemat.

In [21]:
# Clean up the temporary folder
try:
    shutil.rmtree(temp_lfp_folder)
except Exception as e:
    print(f"Could not remove temporary LFP folder {temp_lfp_folder}: {e}")

# Save LFP data to a MATLAB folder
lfp_mat_fname = os.path.join(output_folder, f"imec0_clean_lfp.mat")
scio.savemat(lfp_mat_fname, {
    'depths': final_depths,
    'channel_ids': final_channels,
    'lfp_traces': np.squeeze(np.asarray(final_lfp)),
    'lfp_tvec': final_tvec,
    'lfp_fs': final_fs,
    'shank_ids': shank_ids
})

Could not remove temporary LFP folder D:\preprocessing_sandbox\temp: [WinError 32] The process cannot access the file because it is being used by another process: 'D:\\preprocessing_sandbox\\temp\\traces_cached_seg0.raw'


OverflowError: Python int too large to convert to C long

This is my new attempt to save the LFP data as a .mat file:

In [19]:
# first I'm going to check how many bytes are in the LFP data I'm trying to save
print(f"{np.asarray(final_lfp).nbytes / 1e9:.2f} GB")

6.43 GB


In [21]:
# came out to be ~6.48 GB, which is too big to save with scio.savemat. I might have to use a new library
# called hdf5storage, which can save larger files as a .mat file

# Save the mat file first, while final_lfp/temp_lfp_save are still alive
lfp_mat_fname = os.path.join(output_folder, "imec0_clean_lfp.mat")
hdf5storage.savemat(
    lfp_mat_fname,
    {
        'depths': final_depths,
        'channel_ids': final_channels,
        'lfp_traces': np.squeeze(np.asarray(final_lfp)).astype(np.float32),
        'lfp_tvec': final_tvec,
        'lfp_fs': final_fs,
        'shank_ids': shank_ids
    },
    format='7.3',
    oned_as='row'
)

## Trying to get the final .mat file to have everything formatted the same as before

Ok new problem:  
The above code does save our data as a .mat file, but not all of the data formats are correct in the final .mat file. Specifically:
* imec0.channel_ids is a as a 1x7680 unint32, not n_channels x 64 char
* imec0.depths is a 1 x n_channels cell, not a 1 x n_channels double
* imec0.shank_ids is a 1 x n_channels cell, not a 1 x n_channels int32

imec0.lfp_traces, imec0.lfp_tvec, and imec0.lfp_fs all look great! The only one that is genuinely a problem is the imec0.channel_ids because I cannot parse any of the information. The other ones I would prefer to be fixed so I'm not worried about their comptabilitiy with our later analysis pipelines.

#### Exploring how to do the data structure/type conversions
Starting with final channels (imec0.channel_ids)

In [ ]:
# make the final_channels an array of a different type
print(final_channels.dtype)
print(repr(final_channels[0]))
print(len(final_channels[0]))

shrunk_final_channels = np.array(final_channels.tolist())
print(shrunk_final_channels.dtype)
print(repr(shrunk_final_channels[0]))

trying_another_way = np.asarray(final_channels.tolist(), dtype='U15') # I prefer this because it's more explicit
print(trying_another_way.dtype)

<U64
'imec0.ap#AP0'
12
<U15
'imec0.ap#AP0'
<U15


now we're dealing with depths/final depths

In [ ]:
# now lets deal with the depths/final depths
print(type(final_depths))

<class 'list'>


In [ ]:
# we want it to be a numpy array, so we're going to convert and need it to be a float64 datatype
# so that it will convert to a MATLAB double when saved with hdf5storage.savemat
final_depths_array = np.asarray(final_depths, dtype=np.float64)
print(type(final_depths_array))
print(final_depths_array.dtype)

<class 'numpy.ndarray'>
float64


now we're dealing with shank_ids

In [ ]:
print(type(shank_ids))

<class 'list'>


In [ ]:
# we need it to be int32 dtype in the end, and we want it as an array
shank_ids_array = np.asarray(shank_ids, dtype=np.int32)
print(shank_ids_array.dtype)

int32


#### Doing the final save, with all those conversions implemented
Ok now that we have an idea for how to do all the data type conversions, let's make it into one file save

In [22]:
# Save the mat file first, while final_lfp/temp_lfp_save are still alive
lfp_mat_fname = os.path.join(output_folder, "imec0_clean_lfp.mat")
hdf5storage.savemat(
    lfp_mat_fname,
    {
        'depths': np.asarray(final_depths, dtype=np.float64),
        'channel_ids': np.asarray(final_channels.tolist(), dtype='U15').reshape(-1,1), # added reshape after another formatting failure
        'lfp_traces': np.squeeze(np.asarray(final_lfp, dtype=np.float32)), # I changed this line to match the style im using, but it shouldn't make a functional difference
        'lfp_tvec': final_tvec,
        'lfp_fs': final_fs,
        'shank_ids': np.asarray(shank_ids, dtype=np.int32)
    },
    format='7.3',
    oned_as='row'
)

IT WORKED LETS GOOOOOOOOOOOOOOOO!!!!!

## Trying to get the cleanup to work 
(it doesn't, but it isn't the end of the world because you can just delete it later).

In [ ]:
# First try to delete the variables that might be preventing me from
# deleting the temporary folder 
# del final_lfp, temp_lfp_save, temp_lfp

gc.collect()  # Force garbage collection to free up memory

482

In [ ]:
# Now try to clean up the temporary lfp folder
print(f"Attempting to remove temporary LFP folder: {temp_lfp_folder}")
try:
    shutil.rmtree(temp_lfp_folder)
    print(f"Successfully removed temporary LFP folder: {temp_lfp_folder}")
except Exception as e:
    print(f"Error occurred while trying to remove temporary LFP folder: {e}")

Attempting to remove temporary LFP folder: D:\preprocessing_sandbox\temp
Error occurred while trying to remove temporary LFP folder: [WinError 32] The process cannot access the file because it is being used by another process: 'D:\\preprocessing_sandbox\\temp\\traces_cached_seg0.raw'
